In [132]:
import json
import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import gpt_sample
import naver_stt
from glob import glob
from copy import deepcopy
from functools import reduce
import pandas as pd

STT

In [ ]:
client = naver_stt.ClovaSpeechClient()
error_list=[]
for i in tqdm(range(200)):
    index = "D" + str(i+1)
    file_index = "D" + str(i+1).zfill(3)
    file_path=f'../../data/raw/{file_index}/Voice.m4a'
    try:
        response = client.req_upload(file=file_path, completion='sync')
        if response.status_code == 200:  # 성공적으로 처리된 경우
            response_json = response.json()  # JSON 형식으로 변환
            with open(f"../../data/label/verval_content/verval/{file_index}.txt", "w", encoding="utf-8") as txt_file:
                txt_file.write(response_json['text'])
    except:
        error_list.append(f"{file_path} 없음")

In [ ]:
for i in tqdm(range(200)):
    index = "D" + str(i+1)
    file_index = "D" + str(i+1).zfill(3)
    file_path=f'../../data/raw/{file_index}/Voice.m4a'
    if os.path.exists(file_path):
        continue
    else:
        print(file_path)

GPT

In [ ]:
chat = gpt_sample.chat()
file_list = glob("../../data/label/verval_content/verval/*.txt")
error_list=[]
for i in tqdm(range(len(file_list))):
    try:
        file_index =os.path.splitext(os.path.basename(file_list[i]))[0]
        file_path=f'../../data/label/verval_content/verval/{file_index}.txt'
        with open(file_path, "r", encoding="utf-8") as txt_file:
            text = txt_file.read()

        response=chat.response_data(text)
        json_data =json.loads(response)
        with open(f"../../data/label/verval_content/Analytics_gpt_v.1.0/{file_index}.json", "w",encoding="UTF-8-sig") as json_file:
            json.dump(json_data, json_file, indent="\t", ensure_ascii=False)
            
    except:
        error_list.append(f"{file_path}")
        print(f"{file_path}")

analysis

In [ ]:
predict_list=glob("../../data/label/verval_content/Analytics_gpt_v.1.0/*.json")
label_list=[f.replace("/verval_content/Analytics_gpt_v.1.0","/check_list") for f in predict_list]

In [98]:
with open('./sample.json', "r", encoding="UTF-8-sig") as sample_file:
    sample = json.load(sample_file)['구두']
    
def extract_keys(json_obj, parent_key=""):
    keys = []
    if isinstance(json_obj, dict):
        for key, value in json_obj.items():
            new_key = f"{parent_key}_{key}" if parent_key else key
            keys.append(new_key)
            keys.extend(extract_keys(value, new_key))  # 재귀 호출
    elif isinstance(json_obj, list):
        for i, item in enumerate(json_obj):
            new_key = f"{parent_key}[{i}]"
            keys.append(new_key)
            keys.extend(extract_keys(item, new_key))  # 리스트 항목 탐색
    return keys

tree_list=extract_keys(sample)
for i in range(len(tree_list)):
    key_path = tree_list[i].split('_')  # 경로를 리스트로 변환
        
    # 마지막 키를 제외한 부모 객체 찾기
    parent_keys = key_path[:-1]
    last_key = key_path[-1]

    # parent_dict를 찾아서 최종 키를 접근할 준비
    parent_dict = reduce(lambda d, k: d[k], parent_keys, sample) if parent_keys else sample
    match_index = parent_dict[last_key]

    if isinstance(match_index, dict):  # 값이 딕셔너리라면 스킵
        continue
    match_index =0
    parent_dict[last_key]=match_index
    
tp=deepcopy(sample)
fp=deepcopy(sample)
fn=deepcopy(sample)
tn=deepcopy(sample)   
for i in tqdm(range(len(predict_list))):
    with open(label_list[i], "r", encoding="UTF-8-sig") as label_file:
        label = json.load(label_file)['구두']

    with open(predict_list[i], "r", encoding="UTF-8-sig") as predict_file:
        predict = json.load(predict_file)        
    
    for j in range(len(tree_list)):
        key_path = tree_list[j].split('_')  # 경로를 리스트로 변환
            
        # 마지막 키를 제외한 부모 객체 찾기
        parent_keys = key_path[:-1]
        last_key = key_path[-1]
        
        label_dict = reduce(lambda d, k: d[k], parent_keys, label) if parent_keys else label
        predict_dict = reduce(lambda d, k: d[k], parent_keys, predict) if parent_keys else predict
        match_index = predict_dict[last_key]
        if isinstance(match_index, dict):  # 값이 딕셔너리라면 스킵
            continue
        
        if label_dict[last_key]==predict_dict[last_key]:
            if label_dict[last_key]==True:
                tp_dict = reduce(lambda d, k: d[k], parent_keys, tp) if parent_keys else tp
                value=tp_dict[last_key]
                value=value+1
                tp_dict[last_key]=value
            else:
                tn_dict = reduce(lambda d, k: d[k], parent_keys, tn) if parent_keys else tn
                value=tn_dict[last_key]
                value=value+1
                tn_dict[last_key]=value
        else:
            if label_dict[last_key]==True:
                fp_dict = reduce(lambda d, k: d[k], parent_keys, fp) if parent_keys else fp
                value=fp_dict[last_key]
                value=value+1
                fp_dict[last_key]=value
            else:
                fn_dict = reduce(lambda d, k: d[k], parent_keys, fn) if parent_keys else fn
                value=fn_dict[last_key]
                value=value+1
                fn_dict[last_key]=value




100%|██████████| 194/194 [00:00<00:00, 463.86it/s]


In [ ]:
accuracy=deepcopy(sample)
sensitivity=deepcopy(sample)
specificity=deepcopy(sample)
precision=deepcopy(sample)
f1=deepcopy(sample)
tree_list=extract_keys(sample)
for i in range(len(tree_list)):
    key_path = tree_list[i].split('_')  # 경로를 리스트로 변환
        
    # 마지막 키를 제외한 부모 객체 찾기
    parent_keys = key_path[:-1]
    last_key = key_path[-1]
    
    # parent_dict를 찾아서 최종 키를 접근할 준비
    parent_dict = reduce(lambda d, k: d[k], parent_keys, sample) if parent_keys else sample
    tp_value = reduce(lambda d, k: d[k], parent_keys, tp) if parent_keys else tp
    fp_value = reduce(lambda d, k: d[k], parent_keys, fp) if parent_keys else fp
    fn_value = reduce(lambda d, k: d[k], parent_keys, fn) if parent_keys else fn
    tn_value = reduce(lambda d, k: d[k], parent_keys, tn) if parent_keys else tn
    parent_dict=parent_dict[last_key]
    tp_value=tp_value[last_key]
    fp_value=fp_value[last_key]
    fn_value=fn_value[last_key]
    tn_value=tn_value[last_key]
    if isinstance(parent_dict, dict):  # 값이 딕셔너리라면 스킵
        continue
    accuracy_value=(tp_value+tn_value)/(tp_value+fp_value+fn_value+tn_value)
    sensitivity_value=tp_value/(tp_value+fn_value)
    specificity_value=tn_value/(fp_value+tn_value)
    precision_value=tp_value/(tp_value+fp_value)
    f1_value=2*tp_value/(2*tp_value+fp_value+fn_value)
    
    accuracy_dict=reduce(lambda d, k: d[k], parent_keys, accuracy) if parent_keys else accuracy
    sensitivity_dict=reduce(lambda d, k: d[k], parent_keys, sensitivity) if parent_keys else sensitivity
    specificity_dict=reduce(lambda d, k: d[k], parent_keys, specificity) if parent_keys else specificity
    precision_dict=reduce(lambda d, k: d[k], parent_keys, precision) if parent_keys else precision
    f1_dict=reduce(lambda d, k: d[k], parent_keys, f1) if parent_keys else f1
    accuracy_dict[last_key]=accuracy_value
    sensitivity_dict[last_key]=sensitivity_value
    specificity_dict[last_key]=specificity_value
    precision_dict[last_key]=precision_value
    f1_dict[last_key]=f1_value
with open(f"../../result/verbal/f1.json", "w",encoding="UTF-8-sig") as json_file:
    json.dump(f1, json_file, indent="\t", ensure_ascii=False)
with open(f"../../result/verbal/precision.json", "w",encoding="UTF-8-sig") as json_file:
    json.dump(precision, json_file, indent="\t", ensure_ascii=False)
with open(f"../../result/verbal/sensitivity.json", "w",encoding="UTF-8-sig") as json_file:
    json.dump(sensitivity, json_file, indent="\t", ensure_ascii=False)
with open(f"../../result/verbal/specificity.json", "w",encoding="UTF-8-sig") as json_file:
    json.dump(specificity, json_file, indent="\t", ensure_ascii=False)
with open(f"../../result/verbal/accuracy.json", "w",encoding="UTF-8-sig") as json_file:
    json.dump(accuracy, json_file, indent="\t", ensure_ascii=False)

Figure

In [226]:
df=pd.DataFrame(columns=['번호','평가내용','Accuracy','Sensitivity','Specificity','F1'])
tree_list=extract_keys(sample)
title_number=1
last_number=1
accuracy_list=[]
sensitivity_list=[]
specificity_list=[]
f1_list=[]
for i in range(len(tree_list)):
    key_path = tree_list[i].split('_')  # 경로를 리스트로 변환
    # 마지막 키를 제외한 부모 객체 찾기
    parent_keys = key_path[:-1]
    last_key = key_path[-1]
    accuracy_dict=reduce(lambda d, k: d[k], parent_keys, accuracy) if parent_keys else accuracy
    sensitivity_dict=reduce(lambda d, k: d[k], parent_keys, sensitivity) if parent_keys else sensitivity
    specificity_dict=reduce(lambda d, k: d[k], parent_keys, specificity) if parent_keys else specificity
    precision_dict=reduce(lambda d, k: d[k], parent_keys, precision) if parent_keys else precision
    f1_dict=reduce(lambda d, k: d[k], parent_keys, f1) if parent_keys else f1
    if len(parent_keys)>0: 
        df.loc[i]=[f"{title_number-1}--{last_number}",last_key, accuracy_dict[last_key], sensitivity_dict[last_key], specificity_dict[last_key], f1_dict[last_key]]
        last_number+=1
    elif len(parent_keys)==0 and isinstance(accuracy_dict[last_key], dict):
        df.loc[i]=[title_number, last_key, "", "", "", ""]
        title_number+=1
        last_number=1
    elif len(parent_keys)==0:
        df.loc[i]=[title_number, last_key, accuracy[last_key], sensitivity[last_key], specificity[last_key], f1[last_key]]
        title_number+=1
        last_number=1
    if df.loc[i,"Accuracy"]!="":
        accuracy_list.append(df.loc[i,"Accuracy"])
        sensitivity_list.append(df.loc[i,"Sensitivity"])
        specificity_list.append(df.loc[i,"Specificity"])
        f1_list.append(df.loc[i,"F1"])
        df.loc[i,"Accuracy"]=round(df.loc[i,"Accuracy"],3)
        df.loc[i,"Sensitivity"]=round(df.loc[i,"Sensitivity"],3)
        df.loc[i,"Specificity"]=round(df.loc[i,"Specificity"],3)
        df.loc[i,"F1"]=round(df.loc[i,"F1"],3)
df.loc[i+1]=[title_number, "평균", round(np.mean(accuracy_list),3), round(np.mean(sensitivity_list),3), round(np.mean(specificity_list),3), round(np.mean(f1_list),3)]
df.loc[i+2]=[title_number+1, "표준편차", round(np.std(accuracy_list),3), round(np.std(sensitivity_list),3), round(np.std(specificity_list),3), round(np.std(f1_list),3)]
df.loc[i+3]=[title_number+2, "최소값", round(np.min(accuracy_list),3), round(np.min(sensitivity_list),3), round(np.min(specificity_list),3), round(np.min(f1_list),3)]
df.loc[i+4]=[title_number+3, "최대값", round(np.max(accuracy_list),3), round(np.max(sensitivity_list),3), round(np.max(specificity_list),3), round(np.max(f1_list),3)]
df.to_csv(f"../../result/verbal/verbal.csv", index=False, encoding="utf-8-sig")

In [224]:
accuracy_list

[0.6752577319587629,
 0.7319587628865979,
 0.8762886597938144,
 0.9381443298969072,
 0.9278350515463918,
 0.6958762886597938,
 0.9948453608247423,
 0.9175257731958762,
 0.8556701030927835,
 0.9381443298969072,
 0.9226804123711341,
 0.9896907216494846,
 0.9948453608247423,
 0.9742268041237113,
 0.9484536082474226,
 0.9948453608247423,
 0.9020618556701031,
 0.9381443298969072,
 0.9690721649484536,
 0.9536082474226805,
 0.9690721649484536,
 0.9587628865979382,
 0.8865979381443299,
 0.788659793814433,
 0.8144329896907216,
 0.865979381443299,
 0.7319587628865979,
 0.8402061855670103,
 0.9329896907216495,
 0.9587628865979382]